In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

# Load dataset
df = pd.read_csv("../data/synthetic/users_weight_data.csv")

print(f"TensorFlow version: {tf.__version__}")
print(f"Dataset loaded: {df.shape}")


TensorFlow version: 2.21.0
Dataset loaded: (45000, 15)


In [2]:
# Prepare sequences for LSTM
# Input: 14 days of weight history
# Output: next 7 days of weight

SEQUENCE_LENGTH = 14  # days of history the model looks at
FORECAST_LENGTH = 7   # days ahead the model predicts

def create_sequences(weight_log, seq_len=SEQUENCE_LENGTH, forecast_len=FORECAST_LENGTH):
    X, y = [], []
    for i in range(len(weight_log) - seq_len - forecast_len + 1):
        X.append(weight_log[i : i + seq_len])
        y.append(weight_log[i + seq_len : i + seq_len + forecast_len])
    return np.array(X), np.array(y)

# Build training data from all 500 users
X_all, y_all = [], []

scaler = MinMaxScaler()

for uid in df["user_id"].unique():
    weight_log = df[df["user_id"] == uid]["weight"].values
    
    # Scale each user's weight to 0-1 range
    scaled = scaler.fit_transform(weight_log.reshape(-1, 1)).flatten()
    
    X_user, y_user = create_sequences(scaled)
    X_all.append(X_user)
    y_all.append(y_user)

X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

# Reshape for LSTM — needs (samples, timesteps, features)
X_all = X_all.reshape((X_all.shape[0], X_all.shape[1], 1))

print(f"Training data shape:")
print(f"  X: {X_all.shape}  (samples, timesteps, features)")
print(f"  y: {y_all.shape}  (samples, forecast days)")

# Train/test split — 80/20
split     = int(len(X_all) * 0.8)
X_train   = X_all[:split]
y_train   = y_all[:split]
X_test    = X_all[split:]
y_test    = y_all[split:]

print(f"\nTrain samples: {len(X_train)}")
print(f"Test samples:  {len(X_test)}")

Training data shape:
  X: (35000, 14, 1)  (samples, timesteps, features)
  y: (35000, 7)  (samples, forecast days)

Train samples: 28000
Test samples:  7000


In [3]:
# Build LSTM model
model = Sequential([
    LSTM(64, input_shape=(SEQUENCE_LENGTH, 1), return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(FORECAST_LENGTH)
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 14, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           119 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,959 (117.03 KB)

 Trainable params: 29,959 (117.03 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the model
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print("Training complete")

Epoch 1/50
